# Notebook 01 — NRMS Baseline (O(N²) Self-Attention)

NRMS sử dụng Multi-Head Self-Attention tiêu chuẩn để mô hình hóa:
1. **News Encoder**: Title tokens → Self-Attention → Additive Attention pooling
2. **User Encoder**: History news vectors → Self-Attention → Additive Attention pooling

Độ phức tạp: **O(L² · d)** cho chuỗi lịch sử dài L.

In [ ]:
import sys, time, json, math, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

sys.path.insert(0, '.')
sys.path.append('/kaggle/input/datasets/neitng/utils-for-dl-major-assignment')

from utils import (
    seed_everything, TRAIN_DIR, DEV_DIR, WORK_DIR, MODEL_DIR, SEED,
    load_news, load_behaviors, parse_impressions,
    build_vocab, tokenize_to_ids,
    MINDTrainDataset, collate_train,
    compute_ranking_metrics, count_parameters
)

seed_everything(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'DEVICE: {DEVICE}')

# ── Hyperparameters ────────────────────────────────────────────────────
# Paper NRMS uses pretrained GloVe 300d. Training from scratch
# requires a smaller model, use emb=64, dim=128, heads=8.
EMB_DIM      = 64
NEWS_DIM     = 128
N_HEADS_NEWS = 8
N_HEADS_USER = 8
MAX_TITLE    = 30
MAX_HIST     = 30
NEG_K        = 4
BATCH_SIZE   = 64
LR           = 5e-4
WEIGHT_DECAY = 1e-5
DROPOUT      = 0.1
EPOCHS       = 5
WARMUP_RATIO = 0.1

DEVICE: cuda


In [2]:
print("Loading data...")
news_train = load_news(TRAIN_DIR)
news_dev   = load_news(DEV_DIR)
news_all   = pd.concat([news_train, news_dev]).drop_duplicates('news_id').reset_index(drop=True)

beh_train = load_behaviors(TRAIN_DIR)
beh_dev   = load_behaviors(DEV_DIR)

print(f"News: {len(news_all)}, Train behaviors: {len(beh_train)}, Dev behaviors: {len(beh_dev)}")

Loading data...
News: 65238, Train behaviors: 156965, Dev behaviors: 73152


In [3]:
print("Building vocabulary...")
vocab = build_vocab(news_all, min_freq=2, max_len=MAX_TITLE)
print(f"Vocab size: {len(vocab)}")

# News ID → index (1-based, 0=padding)
all_nids = news_all['news_id'].tolist()
nid2idx = {n: i+1 for i, n in enumerate(all_nids)}

# Pre-tokenize all news titles
news_token_ids = {}
for _, row in news_all.iterrows():
    nidx = nid2idx[row['news_id']]
    text = (row['title'] or '') + ' ' + (row['abstract'] or '')
    news_token_ids[nidx] = tokenize_to_ids(text, vocab, MAX_TITLE)

# Build lookup tensor: shape (num_news+1, MAX_TITLE)
NUM_NEWS = len(nid2idx)
news_tokens_tensor = torch.zeros(NUM_NEWS + 1, MAX_TITLE, dtype=torch.long)
for nidx, tids in news_token_ids.items():
    news_tokens_tensor[nidx] = torch.tensor(tids, dtype=torch.long)

print(f"Tokenized {NUM_NEWS} news articles")

Building vocabulary...
Vocab size: 40689
Tokenized 65238 news articles


In [4]:
class AdditiveAttention(nn.Module):
    """Additive attention pooling: seq → single vector."""
    def __init__(self, dim, hidden=64):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))

    def forward(self, x, mask=None):
        w = self.proj(x).squeeze(-1)                  # (B, L)
        if mask is not None:
            w = w.masked_fill(mask, -1e4)
        w = torch.softmax(w, dim=-1)
        w = torch.nan_to_num(w, nan=0.0)
        return (x * w.unsqueeze(-1)).sum(dim=1)       # (B, D)


class NewsEncoder(nn.Module):
    """Title tokens → news vector via Self-Attention + Additive Attention."""
    def __init__(self, vocab_size, emb_dim, news_dim, n_heads, dropout):
        super().__init__()
        self.word_emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.proj     = nn.Linear(emb_dim, news_dim, bias=False)
        self.norm     = nn.LayerNorm(news_dim)  # stabilise random-init training
        self.dropout  = nn.Dropout(dropout)
        self.mha      = nn.MultiheadAttention(news_dim, n_heads,
                                              batch_first=True, dropout=dropout)
        self.attn_pool = AdditiveAttention(news_dim)

    def forward(self, token_ids):
        mask = (token_ids == 0)                                # True = padding
        
        # Prevent NaN gradients in MHA for all-pad sequences (e.g. dummy news)
        mha_mask = mask.clone()
        all_pad = mha_mask.all(dim=1)
        mha_mask[all_pad, 0] = False
        
        x = self.norm(self.proj(self.word_emb(token_ids)))    # (B, T, D)
        x = self.dropout(x)
        x, _ = self.mha(x, x, x, key_padding_mask=mha_mask)
        x = torch.nan_to_num(x, nan=0.0)
        return self.attn_pool(x, mask)                        # pool with original mask


class UserEncoder(nn.Module):
    """History news vectors → user vector via Self-Attention + Additive Attention."""
    def __init__(self, news_dim, n_heads, dropout):
        super().__init__()
        self.norm     = nn.LayerNorm(news_dim)
        self.mha      = nn.MultiheadAttention(news_dim, n_heads,
                                              batch_first=True, dropout=dropout)
        self.attn_pool = AdditiveAttention(news_dim)

    def forward(self, news_vecs, mask=None):
        mha_mask = mask
        if mask is not None:
            mha_mask = mask.clone()
            all_pad = mha_mask.all(dim=1)
            mha_mask[all_pad, 0] = False

        x = self.norm(news_vecs)                         # normalise before MHA
        x, _ = self.mha(x, x, x, key_padding_mask=mha_mask)
        x = torch.nan_to_num(x, nan=0.0)
        return self.attn_pool(x, mask)


class NRMS(nn.Module):
    def __init__(self, vocab_size, emb_dim, news_dim, n_heads_news,
                 n_heads_user, dropout, news_tokens_lut):
        super().__init__()
        self.news_encoder = NewsEncoder(vocab_size, emb_dim, news_dim,
                                        n_heads_news, dropout)
        self.user_encoder = UserEncoder(news_dim, n_heads_user, dropout)
        # Register lookup table as buffer (not trained)
        self.register_buffer('news_lut', news_tokens_lut)
        self.news_dim = news_dim

    def get_news_tokens(self, nid_indices):
        """Lookup pre-tokenized news by index."""
        return self.news_lut[nid_indices]  # (*, MAX_TITLE)

    def encode_news(self, nid_indices):
        """Encode news items by their index."""
        tokens = self.get_news_tokens(nid_indices)
        shape = tokens.shape
        if len(shape) > 2:
            flat = tokens.view(-1, shape[-1])
            vecs = self.news_encoder(flat)
            return vecs.view(*shape[:-1], -1)
        return self.news_encoder(tokens)

    def encode_user(self, hist_indices):
        """hist_indices: (B, L) news index IDs."""
        mask = (hist_indices == 0)
        news_vecs = self.encode_news(hist_indices)  # (B, L, D)
        return self.user_encoder(news_vecs, mask)    # (B, D)

    def forward(self, hist, pos, neg):
        """
        hist: (B, L) history news indices
        pos:  (B,) positive candidate index
        neg:  (B, K) negative candidate indices
        Returns: logits (B, 1+K)
        """
        u = self.encode_user(hist)             # (B, D)
        p_vec = self.encode_news(pos)          # (B, D)
        n_vec = self.encode_news(neg)          # (B, K, D)
        pos_score = (u * p_vec).sum(-1, keepdim=True)        # (B, 1)
        neg_score = (u.unsqueeze(1) * n_vec).sum(-1)         # (B, K)
        return torch.cat([pos_score, neg_score], dim=1)      # (B, 1+K)

    def score_candidates(self, hist, cand_indices):
        """Score each candidate for evaluation."""
        u = self.encode_user(hist)             # (1, D)
        c_vec = self.encode_news(cand_indices) # (C, D)
        return (u * c_vec).sum(-1)             # (C,)

In [5]:
model = NRMS(
    vocab_size=len(vocab),
    emb_dim=EMB_DIM,
    news_dim=NEWS_DIM,
    n_heads_news=N_HEADS_NEWS,
    n_heads_user=N_HEADS_USER,
    dropout=DROPOUT,
    news_tokens_lut=news_tokens_tensor,
).to(DEVICE)

n_params = count_parameters(model)
print(f"NRMS parameters: {n_params:,} ({n_params/1e6:.2f}M)")

NRMS parameters: 2,761,538 (2.76M)


In [6]:
print("Building training dataset...")
train_ds = MINDTrainDataset(beh_train, nid2idx, max_hist=MAX_HIST,
                            neg_k=NEG_K, max_rows=150000)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                      collate_fn=lambda b: collate_train(b, MAX_HIST),
                      num_workers=2, pin_memory=True)
print(f"Training samples: {len(train_ds)}")

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
loss_fn = nn.CrossEntropyLoss()  # target = 0 (pos is at index 0)

# Linear warmup + cosine decay scheduler
total_steps = len(train_dl) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

epoch_times = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    losses = []
    t0 = time.time()
    for H, P, N in train_dl:
        H, P, N = H.to(DEVICE), P.to(DEVICE), N.to(DEVICE)
        optimizer.zero_grad()
        logits = model(H, P, N)  # (B, 1+K)
        target = torch.zeros(logits.size(0), dtype=torch.long, device=DEVICE)
        loss = loss_fn(logits, target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        losses.append(loss.item())

    dt = time.time() - t0
    epoch_times.append(dt)
    print(f"Epoch {epoch}/{EPOCHS} | loss={np.mean(losses):.5f} | time={dt:.1f}s")

Building training dataset...
Training samples: 150000
Epoch 1/5 | loss=1.50133 | time=113.9s
Epoch 2/5 | loss=1.39656 | time=114.4s
Epoch 3/5 | loss=1.34677 | time=115.0s
Epoch 4/5 | loss=1.30426 | time=115.2s
Epoch 5/5 | loss=1.27620 | time=115.7s


In [7]:
print("\nEvaluating on dev set...")
from utils import evaluate_model
metrics = evaluate_model(model, beh_dev, nid2idx, news_token_ids,
                         max_hist=MAX_HIST, device=DEVICE)
print("=" * 50)
print("NRMS Baseline Results:")
for k, v in metrics.items():
    print(f"  {k}: {v:.6f}")
print("=" * 50)


Evaluating on dev set...
NRMS Baseline Results:
  AUC: 0.615643
  MRR: 0.325132
  nDCG@5: 0.309666
  nDCG@10: 0.372126


In [8]:
results = {
    'model': 'NRMS',
    'type': 'baseline',
    'complexity': 'O(L^2 * d)',
    'parameters': n_params,
    'metrics': metrics,
    'epoch_times_sec': epoch_times,
    'avg_epoch_time_sec': float(np.mean(epoch_times)),
    'hyperparams': {
        'emb_dim': EMB_DIM, 'news_dim': NEWS_DIM,
        'n_heads_news': N_HEADS_NEWS, 'n_heads_user': N_HEADS_USER,
        'max_title': MAX_TITLE, 'max_hist': MAX_HIST,
        'neg_k': NEG_K, 'batch_size': BATCH_SIZE,
        'lr': LR, 'epochs': EPOCHS, 'dropout': DROPOUT,
    }
}

torch.save(model.state_dict(), MODEL_DIR / 'nrms_baseline.pt')
with open(WORK_DIR / 'nrms_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"\nModel saved to {MODEL_DIR / 'nrms_baseline.pt'}")
print(f"Results saved to {WORK_DIR / 'nrms_results.json'}")


Model saved to /kaggle/working/dl_results/models/nrms_baseline.pt
Results saved to /kaggle/working/dl_results/nrms_results.json
